# Эксперимент 7 (Трек 2): Предобуславливание HFN + оценка Хатчинсона

In [ ]:
import time
import urllib.request
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import scipy.sparse as sp
from sklearn.datasets import load_svmlight_file

from oracles import LogCoshL2Oracle, ExponentialLossL2Oracle
from utils import get_line_search_tool

In [ ]:
LIBSVM_BASE = "https://www.csie.ntu.edu.tw/~cjlin/libsvmtools/datasets"
data_dir = Path("data/libsvm")
data_dir.mkdir(parents=True, exist_ok=True)

def ensure_dataset(name, rel_url):
    path = data_dir / name
    if not path.exists():
        req = urllib.request.Request(f"{LIBSVM_BASE}/{rel_url}", headers={"User-Agent": "MetOpt-lab/2.0"})
        with urllib.request.urlopen(req, timeout=300) as r, open(path, "wb") as f:
            f.write(r.read())
    return path

def labels_pm_one(y):
    y = np.asarray(y, dtype=float).ravel()
    u = np.unique(y)
    if len(u) == 2 and 0.0 in u and 1.0 in u:
        return 2.0 * y - 1.0
    return y

def sparse_oracle_ops(X):
    if not sp.issparse(X):
        X = sp.csr_matrix(X, dtype=np.float64)
    else:
        X = X.astype(np.float64).tocsr()
    def matvec_Ax(x):
        return X.dot(np.asarray(x, dtype=np.float64).ravel())
    def matvec_ATx(v):
        return X.T.dot(np.asarray(v, dtype=np.float64).ravel())
    def matmat_ATsA(s):
        s = np.asarray(s, dtype=np.float64).ravel()
        return X.T.dot(sp.diags(s).dot(X)).toarray()
    return matvec_Ax, matvec_ATx, matmat_ATsA

clf_path = ensure_dataset("a9a", "binary/a9a")
X, y = load_svmlight_file(clf_path)
X = X.tocsr().astype(np.float64)
y = labels_pm_one(y)
m, n = X.shape
print(f"dataset shape: m={m}, n={n}")

In [ ]:
# Искажение обусловленности: 1-я половина *1000, 2-я половина *0.001
X_bad = X.copy().tolil()
half = n // 2
if half > 0:
    X_bad[:, :half] = X_bad[:, :half] * 1000.0
    X_bad[:, half:] = X_bad[:, half:] * 0.001
X_bad = X_bad.tocsr()

regcoef = 1.0 / m
oracle = ExponentialLossL2Oracle(*sparse_oracle_ops(X_bad), y, regcoef)
x0 = np.zeros(n)

In [ ]:
def hutchinson_diag_estimator(oracle, x, N=15, seed=42):
    rng = np.random.default_rng(seed)
    n = x.size
    acc = np.zeros(n, dtype=np.float64)
    for _ in range(N):
        z = rng.choice([-1.0, 1.0], size=n)
        Hz = oracle.hess_vec(x, z)
        acc += z * Hz
    v = acc / max(N, 1)
    M_diag = np.maximum(np.abs(v), 1e-6)
    return M_diag

def pcg(matvec, b, x0, tol=1e-4, max_iter=None, M_diag=None):
    x = x0.copy()
    n = b.size
    if max_iter is None:
        max_iter = n
    r = b - matvec(x)
    b_norm = np.linalg.norm(b)
    if b_norm == 0:
        b_norm = 1.0
    if np.linalg.norm(r) <= tol * b_norm:
        return x, 0
    if M_diag is None:
        z = r.copy()
    else:
        z = r / M_diag
    p = z.copy()
    rz_old = np.dot(r, z)

    for k in range(1, max_iter + 1):
        Ap = matvec(p)
        denom = np.dot(p, Ap)
        if denom <= 1e-20:
            return x, k
        alpha = rz_old / denom
        x = x + alpha * p
        r = r - alpha * Ap
        if np.linalg.norm(r) <= tol * b_norm:
            return x, k
        if M_diag is None:
            z = r.copy()
        else:
            z = r / M_diag
        rz_new = np.dot(r, z)
        beta = rz_new / max(rz_old, 1e-32)
        p = z + beta * p
        rz_old = rz_new
    return x, max_iter

In [ ]:
def hfn_run(oracle, x0, tol=1e-8, max_iter=40, line_search_options=None, use_precond=False, hutchinson_N=15):
    line_search_tool = get_line_search_tool(line_search_options)
    x = x0.copy()
    g = oracle.grad(x)
    g0_sq = np.dot(g, g)
    if g0_sq == 0:
        g0_sq = 1.0
    hist = {"time": [], "func": [], "grad_norm": [], "cg_iters": []}
    t0 = time.perf_counter()

    for _ in range(max_iter):
        hist["time"].append(time.perf_counter() - t0)
        hist["func"].append(oracle.func(x))
        hist["grad_norm"].append(np.linalg.norm(g))
        if np.dot(g, g) <= tol * g0_sq:
            break

        eta = min(0.5, np.sqrt(np.linalg.norm(g)))
        b_cg = -g
        hv = lambda v: oracle.hess_vec(x, v)

        M_diag = None
        if use_precond:
            M_diag = hutchinson_diag_estimator(oracle, x, N=hutchinson_N)

        d_init = -g
        while True:
            d, cg_it = pcg(hv, b_cg, x0=d_init, tol=eta, max_iter=None, M_diag=M_diag)
            if np.dot(g, d) < 0:
                hist["cg_iters"].append(cg_it)
                break
            eta *= 0.1
            d_init = d

        alpha = line_search_tool.line_search(oracle, x, d, previous_alpha=1.0)
        if alpha is None or alpha <= 0:
            alpha = 1e-4
        x = x + alpha * d
        g = oracle.grad(x)

    return x, hist

ls = {"method": "Wolfe", "c1": 1e-4, "c2": 0.9, "alpha_0": 1.0}

In [ ]:
# Базовый HFN
x_base, h_base = hfn_run(oracle, x0, tol=1e-8, max_iter=35, line_search_options=ls, use_precond=False)

# Предобусловленный HFN (несколько N для баланса)
Ns = [5, 10, 20]
precond_runs = {}
for N in Ns:
    xN, hN = hfn_run(oracle, x0, tol=1e-8, max_iter=35, line_search_options=ls, use_precond=True, hutchinson_N=N)
    precond_runs[N] = (xN, hN)

print("done")

In [ ]:
def rel_grad_sq(hist):
    g = np.array(hist["grad_norm"])
    return (g ** 2) / max(g[0] ** 2, 1e-32)

plt.figure(figsize=(14, 4))

plt.subplot(1, 2, 1)
plt.plot(rel_grad_sq(h_base), label="HFN base")
for N, (_, hN) in precond_runs.items():
    plt.plot(rel_grad_sq(hN), label=f"HFN+Prec (N={N})")
plt.yscale("log")
plt.xlabel("outer iteration")
plt.ylabel(r"$||g_k||^2 / ||g_0||^2$")
plt.title("Ошибка vs внешние итерации")
plt.grid(True, alpha=0.3)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(h_base["time"], rel_grad_sq(h_base), label="HFN base")
for N, (_, hN) in precond_runs.items():
    plt.plot(hN["time"], rel_grad_sq(hN), label=f"HFN+Prec (N={N})")
plt.yscale("log")
plt.xlabel("time, sec")
plt.ylabel(r"$||g_k||^2 / ||g_0||^2$")
plt.title("Ошибка vs время")
plt.grid(True, alpha=0.3)
plt.legend()

plt.tight_layout()
Path('fig').mkdir(parents=True, exist_ok=True)
plt.savefig('fig/exp7_hfn_error_curves.png', dpi=200, bbox_inches='tight')

In [ ]:
plt.figure(figsize=(14, 4))

plt.subplot(1, 2, 1)
plt.plot(h_base["cg_iters"], marker="o", label="HFN base")
for N, (_, hN) in precond_runs.items():
    plt.plot(hN["cg_iters"], marker="o", label=f"HFN+Prec (N={N})")
plt.xlabel("outer iteration")
plt.ylabel("CG iterations")
plt.title("Внутренние итерации CG")
plt.grid(True, alpha=0.3)
plt.legend()

plt.subplot(1, 2, 2)
labels = ["base"] + [f"N={N}" for N in Ns]
vals = [h_base["time"][-1]] + [precond_runs[N][1]["time"][-1] for N in Ns]
plt.bar(labels, vals)
plt.ylabel("total time, sec")
plt.title("Итоговое время (баланс затрат Hutchinson)")
plt.grid(True, axis="y", alpha=0.3)

plt.tight_layout()
Path('fig').mkdir(parents=True, exist_ok=True)
plt.savefig('fig/exp7_hfn_cg_and_time.png', dpi=200, bbox_inches='tight')